In [1]:
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
import os


# Подготовка датасета

In [2]:
file_path = './MixtureSolDB_with_features_small.csv'

df = pd.read_csv(file_path).dropna().reset_index(drop=True)

In [3]:
df

,Compound_Name,SMILES_Solute,Temperature_K,Solubility(mole_fraction),LogS(mole_fraction),Solvent1,Solvent2,SMILES_Solvent1,SMILES_Solvent2,x1,...,S2_NumHAcceptors,S2_NumRotatableBonds,S2_RingCount,S2_NumAromaticRings,S2_FractionCSP3,S2_HeavyAtomCount,S2_BertzCT,S2_BalabanJ,S2_MaxPartialCharge,S2_MinPartialCharge
0,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,278.15,0.0872,-1.059484,acetonitrile,water,CC#N,O,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
1,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,283.15,0.0983,-1.007446,acetonitrile,water,CC#N,O,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
2,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,288.15,0.1126,-0.948462,acetonitrile,water,CC#N,O,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
3,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,293.15,0.1287,-0.890421,acetonitrile,water,CC#N,O,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
4,Rivastigmine tartrate,CCN(C)C(=O)Oc1cccc([C@H](C)N(C)C)c1.O=C(O)[C@H...,298.15,0.1469,-0.832978,acetonitrile,water,CC#N,O,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117143,Glutaric acid,O=C(O)CCCC(=O)O,303.15,0.2683,-0.571379,ethanol,water,CCO,O,1.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117144,Glutaric acid,O=C(O)CCCC(=O)O,305.15,0.2769,-0.557677,ethanol,water,CCO,O,1.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117145,Glutaric acid,O=C(O)CCCC(=O)O,308.15,0.2964,-0.528122,ethanol,water,CCO,O,1.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117146,Glutaric acid,O=C(O)CCCC(=O)O,310.15,0.3081,-0.511308,ethanol,water,CCO,O,1.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151


In [4]:
df.columns

Index(['Compound_Name', 'SMILES_Solute', 'Temperature_K',
       'Solubility(mole_fraction)', 'LogS(mole_fraction)', 'Solvent1',
       'Solvent2', 'SMILES_Solvent1', 'SMILES_Solvent2', 'x1', 'x2',
       'IsPureSolventEndpoint', 'S1_pure_solubility', 'S1_pure_logS',
       'S1_temp_diff', 'S2_pure_solubility', 'S2_pure_logS', 'S2_temp_diff',
       'solute_MolWt', 'solute_MolLogP', 'solute_TPSA', 'solute_NumHDonors',
       'solute_NumHAcceptors', 'solute_NumRotatableBonds', 'solute_RingCount',
       'solute_NumAromaticRings', 'solute_FractionCSP3',
       'solute_HeavyAtomCount', 'solute_BertzCT', 'solute_BalabanJ',
       'solute_MaxPartialCharge', 'solute_MinPartialCharge', 'S1_MolWt',
       'S1_MolLogP', 'S1_TPSA', 'S1_NumHDonors', 'S1_NumHAcceptors',
       'S1_NumRotatableBonds', 'S1_RingCount', 'S1_NumAromaticRings',
       'S1_FractionCSP3', 'S1_HeavyAtomCount', 'S1_BertzCT', 'S1_BalabanJ',
       'S1_MaxPartialCharge', 'S1_MinPartialCharge', 'S2_MolWt', 'S2_MolLogP',
      

In [5]:
target_col = 'LogS(mole_fraction)'

drop_cols = [
    # target_col,
    'Solubility(mole_fraction)',
    'Compound_Name',
    'Solvent1',
    'Solvent2',
    'SMILES_Solute',
    'SMILES_Solvent1',
    'SMILES_Solvent2',
    
    'IsPureSolventEndpoint'

]

feature_cols = [c for c in df.columns if c not in drop_cols]

print("Number of features:", len(feature_cols))
print(feature_cols[:])

Number of features: 52
['Temperature_K', 'LogS(mole_fraction)', 'x1', 'x2', 'S1_pure_solubility', 'S1_pure_logS', 'S1_temp_diff', 'S2_pure_solubility', 'S2_pure_logS', 'S2_temp_diff', 'solute_MolWt', 'solute_MolLogP', 'solute_TPSA', 'solute_NumHDonors', 'solute_NumHAcceptors', 'solute_NumRotatableBonds', 'solute_RingCount', 'solute_NumAromaticRings', 'solute_FractionCSP3', 'solute_HeavyAtomCount', 'solute_BertzCT', 'solute_BalabanJ', 'solute_MaxPartialCharge', 'solute_MinPartialCharge', 'S1_MolWt', 'S1_MolLogP', 'S1_TPSA', 'S1_NumHDonors', 'S1_NumHAcceptors', 'S1_NumRotatableBonds', 'S1_RingCount', 'S1_NumAromaticRings', 'S1_FractionCSP3', 'S1_HeavyAtomCount', 'S1_BertzCT', 'S1_BalabanJ', 'S1_MaxPartialCharge', 'S1_MinPartialCharge', 'S2_MolWt', 'S2_MolLogP', 'S2_TPSA', 'S2_NumHDonors', 'S2_NumHAcceptors', 'S2_NumRotatableBonds', 'S2_RingCount', 'S2_NumAromaticRings', 'S2_FractionCSP3', 'S2_HeavyAtomCount', 'S2_BertzCT', 'S2_BalabanJ', 'S2_MaxPartialCharge', 'S2_MinPartialCharge']


In [6]:
x_vars = []
solute_vars = []
free_vars = []
for col in feature_cols:
    if col.startswith('S1_'):
        name = col[3:]
        if name not in x_vars:
            x_vars.append(name)
            assert f'S2_{name}' in feature_cols

    elif col.startswith('S2_'):
        name = col[3:]
        if name not in x_vars:
            x_vars.append(name)
            assert f'S1_{name}' in feature_cols

    elif col.startswith('solute_'):
        solute_vars.append(col)
    else:
        free_vars.append(col)
        


In [7]:
x_vars, solute_vars, free_vars

(['pure_solubility',
  'pure_logS',
  'temp_diff',
  'MolWt',
  'MolLogP',
  'TPSA',
  'NumHDonors',
  'NumHAcceptors',
  'NumRotatableBonds',
  'RingCount',
  'NumAromaticRings',
  'FractionCSP3',
  'HeavyAtomCount',
  'BertzCT',
  'BalabanJ',
  'MaxPartialCharge',
  'MinPartialCharge'],
 ['solute_MolWt',
  'solute_MolLogP',
  'solute_TPSA',
  'solute_NumHDonors',
  'solute_NumHAcceptors',
  'solute_NumRotatableBonds',
  'solute_RingCount',
  'solute_NumAromaticRings',
  'solute_FractionCSP3',
  'solute_HeavyAtomCount',
  'solute_BertzCT',
  'solute_BalabanJ',
  'solute_MaxPartialCharge',
  'solute_MinPartialCharge'],
 ['Temperature_K', 'LogS(mole_fraction)', 'x1', 'x2'])

In [8]:
df2 = df[feature_cols]
df2

,Temperature_K,LogS(mole_fraction),x1,x2,S1_pure_solubility,S1_pure_logS,S1_temp_diff,S2_pure_solubility,S2_pure_logS,S2_temp_diff,...,S2_NumHAcceptors,S2_NumRotatableBonds,S2_RingCount,S2_NumAromaticRings,S2_FractionCSP3,S2_HeavyAtomCount,S2_BertzCT,S2_BalabanJ,S2_MaxPartialCharge,S2_MinPartialCharge
0,278.15,-1.059484,0.0,1.0,0.0207,-1.684030,0.0,0.0872,-1.059484,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
1,283.15,-1.007446,0.0,1.0,0.0241,-1.617983,0.0,0.0983,-1.007446,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
2,288.15,-0.948462,0.0,1.0,0.0280,-1.552842,0.0,0.1126,-0.948462,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
3,293.15,-0.890421,0.0,1.0,0.0321,-1.493495,0.0,0.1287,-0.890421,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
4,298.15,-0.832978,0.0,1.0,0.0372,-1.429457,0.0,0.1469,-0.832978,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117143,303.15,-0.571379,1.0,0.0,0.2683,-0.571379,0.0,0.1845,-0.734004,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117144,305.15,-0.557677,1.0,0.0,0.2769,-0.557677,0.0,0.1819,-0.740167,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117145,308.15,-0.528122,1.0,0.0,0.2964,-0.528122,0.0,0.1976,-0.704213,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117146,310.15,-0.511308,1.0,0.0,0.3081,-0.511308,0.0,0.1915,-0.717831,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151


In [9]:
idx_pure = df2[df2['x1'].isin([0, 1])].index
idx_mix = df2[~df2['x1'].isin([0, 1])].index
idx_pure 

Index([     0,      1,      2,      3,      4,      5,      6,      7,      8,
            9,
       ...
       117057, 117139, 117140, 117141, 117142, 117143, 117144, 117145, 117146,
       117147],
      dtype='int64', length=24231)

In [21]:
df2[target_col].iloc[idx_pure]

0        -1.059484
1        -1.007446
2        -0.948462
3        -0.890421
4        -0.832978
            ...   
117143   -0.571379
117144   -0.557677
117145   -0.528122
117146   -0.511308
117147   -0.484126
Name: LogS(mole_fraction), Length: 24231, dtype: float64

In [11]:
df2.iloc[idx_pure]

,Temperature_K,LogS(mole_fraction),x1,x2,S1_pure_solubility,S1_pure_logS,S1_temp_diff,S2_pure_solubility,S2_pure_logS,S2_temp_diff,...,S2_NumHAcceptors,S2_NumRotatableBonds,S2_RingCount,S2_NumAromaticRings,S2_FractionCSP3,S2_HeavyAtomCount,S2_BertzCT,S2_BalabanJ,S2_MaxPartialCharge,S2_MinPartialCharge
0,278.15,-1.059484,0.0,1.0,0.0207,-1.684030,0.0,0.0872,-1.059484,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
1,283.15,-1.007446,0.0,1.0,0.0241,-1.617983,0.0,0.0983,-1.007446,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
2,288.15,-0.948462,0.0,1.0,0.0280,-1.552842,0.0,0.1126,-0.948462,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
3,293.15,-0.890421,0.0,1.0,0.0321,-1.493495,0.0,0.1287,-0.890421,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
4,298.15,-0.832978,0.0,1.0,0.0372,-1.429457,0.0,0.1469,-0.832978,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117143,303.15,-0.571379,1.0,0.0,0.2683,-0.571379,0.0,0.1845,-0.734004,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117144,305.15,-0.557677,1.0,0.0,0.2769,-0.557677,0.0,0.1819,-0.740167,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117145,308.15,-0.528122,1.0,0.0,0.2964,-0.528122,0.0,0.1976,-0.704213,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
117146,310.15,-0.511308,1.0,0.0,0.3081,-0.511308,0.0,0.1915,-0.717831,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151


In [12]:
if False:
    train_frac = 0.8 # Пожалеем данные, их и так мало
    
    N_mix = len(idx_mix)
    print(f"total mix: {N_mix}")
    idx_train = np.random.choice(idx_mix, int(N_mix*train_frac), replace=False)
    idx_test = np.setdiff1d(idx_mix, idx_train)
    # Проверка
    (sorted(np.concat([idx_train, idx_test])) == idx_mix).all()
    idx_train = np.concat([idx_train, idx_pure])
    np.random.shuffle(idx_train)

In [13]:
groups = df['SMILES_Solute'].iloc[idx_mix]
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss1.split(idx_mix, groups=groups))
idx_test = np.array(idx_mix[test_idx])
idx_train = np.array(idx_mix[train_idx])
# idx_train = np.concat([idx_mix[train_idx], idx_pure])
np.random.shuffle(idx_train)
# (sorted(np.concat([idx_train, idx_test])) == np.arange(len(idx_train) + len(idx_test)) ).all()

In [14]:
df2.iloc[idx_train].reset_index(drop=True)

,Temperature_K,LogS(mole_fraction),x1,x2,S1_pure_solubility,S1_pure_logS,S1_temp_diff,S2_pure_solubility,S2_pure_logS,S2_temp_diff,...,S2_NumHAcceptors,S2_NumRotatableBonds,S2_RingCount,S2_NumAromaticRings,S2_FractionCSP3,S2_HeavyAtomCount,S2_BertzCT,S2_BalabanJ,S2_MaxPartialCharge,S2_MinPartialCharge
0,303.15,-3.888401,0.436900,0.563100,0.000022,-4.661544,0.0,1.284200e-03,-2.891367,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
1,298.15,-0.896881,0.099027,0.900973,0.376200,-0.424581,0.0,4.990000e-02,-1.301899,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
2,298.15,-5.667158,0.069718,0.930282,0.000237,-3.625618,0.0,1.256000e-07,-6.901010,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
3,298.15,-2.181774,0.720000,0.280000,0.005814,-2.235525,0.0,2.312000e-03,-2.636012,0.0,...,2,1,0,0,0.75,6,49.509775,2.831257,0.302113,-0.466172
4,278.15,-2.562249,0.316000,0.684000,0.012300,-1.910095,0.0,5.520000e-05,-4.258061,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72055,298.15,-2.275724,0.200000,0.800000,0.000390,-3.408935,0.0,3.660000e-03,-2.436519,0.0,...,1,2,0,0,1.00,5,11.119415,2.190610,0.043042,-0.396377
72056,298.15,-3.871601,0.089055,0.910945,0.013360,-1.874194,0.0,4.645000e-05,-4.333014,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
72057,313.15,-1.851552,0.432200,0.567800,0.031488,-1.501855,0.0,8.210000e-04,-3.085657,0.0,...,0,3,0,0,1.00,6,12.000000,2.339092,-0.053579,-0.065382
72058,293.15,-2.674074,0.691886,0.308114,0.002317,-2.635074,0.0,1.078000e-04,-3.967381,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510


In [15]:
df2.iloc[idx_test].reset_index(drop=True)

,Temperature_K,LogS(mole_fraction),x1,x2,S1_pure_solubility,S1_pure_logS,S1_temp_diff,S2_pure_solubility,S2_pure_logS,S2_temp_diff,...,S2_NumHAcceptors,S2_NumRotatableBonds,S2_RingCount,S2_NumAromaticRings,S2_FractionCSP3,S2_HeavyAtomCount,S2_BertzCT,S2_BalabanJ,S2_MaxPartialCharge,S2_MinPartialCharge
0,283.15,-5.275560,0.022214,0.977786,0.1201,-0.920457,0.0,3.170000e-07,-6.498941,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
1,288.15,-5.162348,0.022214,0.977786,0.1392,-0.856361,0.0,4.024000e-07,-6.395342,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
2,293.15,-5.047498,0.022214,0.977786,0.1604,-0.794796,0.0,5.234000e-07,-6.281166,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
3,298.15,-4.938548,0.022214,0.977786,0.1845,-0.734004,0.0,6.683000e-07,-6.175029,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
4,303.15,-4.830914,0.022214,0.977786,0.2102,-0.677367,0.0,8.379000e-07,-6.076808,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20852,298.15,-1.675718,0.500000,0.500000,0.0275,-1.560667,0.0,1.861143e-04,-3.731865,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
20853,298.15,-1.595166,0.600000,0.400000,0.0275,-1.560667,0.0,1.861143e-04,-3.731865,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
20854,298.15,-1.518557,0.700000,0.300000,0.0275,-1.560667,0.0,1.861143e-04,-3.731865,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
20855,298.15,-1.454693,0.800000,0.200000,0.0275,-1.560667,0.0,1.861143e-04,-3.731865,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151


In [16]:
def expand_dataset(df, pair=x_vars):
    """
    Расширяет датасет, добавляя симметричные строки с перестановкой S1<->S2 и x1<->x2.
    
    Parameters:
    df : pandas.DataFrame
        Исходный датасет.
    pair : list
        Список базовых имён признаков, для которых существуют колонки 'S1_*' и 'S2_*'.
    
    Returns:
    pandas.DataFrame
        Расширенный датасет без дубликатов.
    """
    # Списки имён колонок
    s1_cols = [f'S1_{p}' for p in pair]
    s2_cols = [f'S2_{p}' for p in pair]
    
    # 1. Создаём переставленный DataFrame (копия)
    df_swapped = df.copy()
    
    # 2. Меняем местами S1_* и S2_*
    df_swapped[s1_cols] = df[s2_cols].values
    df_swapped[s2_cols] = df[s1_cols].values
    
    # 3. Меняем местами x1 и x2
    df_swapped['x1'], df_swapped['x2'] = df['x2'], df['x1']
    
    # 4. Приводим порядок колонок к исходному (для корректного merge)
    df_swapped = df_swapped[df.columns]
    
    # 5. Оставляем только те строки, которых нет в исходном df
    #    Делаем left merge и фильтруем по '_merge' == 'left_only'
    merged = df_swapped.merge(df, how='left', indicator=True)
    df_new = merged[merged['_merge'] == 'left_only'].drop('_merge', axis=1)
    
    # 6. Объединяем исходный и новые строки
    df_extended = pd.concat([df, df_new], ignore_index=True)
    
    return df_extended

In [17]:
df_train = df2.iloc[idx_train].reset_index(drop=True)
df_train

,Temperature_K,LogS(mole_fraction),x1,x2,S1_pure_solubility,S1_pure_logS,S1_temp_diff,S2_pure_solubility,S2_pure_logS,S2_temp_diff,...,S2_NumHAcceptors,S2_NumRotatableBonds,S2_RingCount,S2_NumAromaticRings,S2_FractionCSP3,S2_HeavyAtomCount,S2_BertzCT,S2_BalabanJ,S2_MaxPartialCharge,S2_MinPartialCharge
0,303.15,-3.888401,0.436900,0.563100,0.000022,-4.661544,0.0,1.284200e-03,-2.891367,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
1,298.15,-0.896881,0.099027,0.900973,0.376200,-0.424581,0.0,4.990000e-02,-1.301899,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
2,298.15,-5.667158,0.069718,0.930282,0.000237,-3.625618,0.0,1.256000e-07,-6.901010,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
3,298.15,-2.181774,0.720000,0.280000,0.005814,-2.235525,0.0,2.312000e-03,-2.636012,0.0,...,2,1,0,0,0.75,6,49.509775,2.831257,0.302113,-0.466172
4,278.15,-2.562249,0.316000,0.684000,0.012300,-1.910095,0.0,5.520000e-05,-4.258061,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72055,298.15,-2.275724,0.200000,0.800000,0.000390,-3.408935,0.0,3.660000e-03,-2.436519,0.0,...,1,2,0,0,1.00,5,11.119415,2.190610,0.043042,-0.396377
72056,298.15,-3.871601,0.089055,0.910945,0.013360,-1.874194,0.0,4.645000e-05,-4.333014,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510
72057,313.15,-1.851552,0.432200,0.567800,0.031488,-1.501855,0.0,8.210000e-04,-3.085657,0.0,...,0,3,0,0,1.00,6,12.000000,2.339092,-0.053579,-0.065382
72058,293.15,-2.674074,0.691886,0.308114,0.002317,-2.635074,0.0,1.078000e-04,-3.967381,0.0,...,0,0,0,0,0.00,1,0.000000,0.000000,-0.411510,-0.411510


In [18]:
df_train_ex = expand_dataset(df_train)
df_train_ex

,Temperature_K,LogS(mole_fraction),x1,x2,S1_pure_solubility,S1_pure_logS,S1_temp_diff,S2_pure_solubility,S2_pure_logS,S2_temp_diff,...,S2_NumHAcceptors,S2_NumRotatableBonds,S2_RingCount,S2_NumAromaticRings,S2_FractionCSP3,S2_HeavyAtomCount,S2_BertzCT,S2_BalabanJ,S2_MaxPartialCharge,S2_MinPartialCharge
0,303.15,-3.888401,0.436900,0.563100,0.000022,-4.661544,0.0,1.284200e-03,-2.891367,0.0,...,0.0,0.0,0.0,0.0,0.000000,1.0,0.000000,0.000000,-0.411510,-0.411510
1,298.15,-0.896881,0.099027,0.900973,0.376200,-0.424581,0.0,4.990000e-02,-1.301899,0.0,...,0.0,0.0,0.0,0.0,0.000000,1.0,0.000000,0.000000,-0.411510,-0.411510
2,298.15,-5.667158,0.069718,0.930282,0.000237,-3.625618,0.0,1.256000e-07,-6.901010,0.0,...,0.0,0.0,0.0,0.0,0.000000,1.0,0.000000,0.000000,-0.411510,-0.411510
3,298.15,-2.181774,0.720000,0.280000,0.005814,-2.235525,0.0,2.312000e-03,-2.636012,0.0,...,2.0,1.0,0.0,0.0,0.750000,6.0,49.509775,2.831257,0.302113,-0.466172
4,278.15,-2.562249,0.316000,0.684000,0.012300,-1.910095,0.0,5.520000e-05,-4.258061,0.0,...,0.0,0.0,0.0,0.0,0.000000,1.0,0.000000,0.000000,-0.411510,-0.411510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144115,298.15,-2.275724,0.800000,0.200000,0.003660,-2.436519,0.0,3.900000e-04,-3.408935,0.0,...,1.0,0.0,0.0,0.0,0.500000,3.0,24.264663,2.475534,0.058715,-0.198658
144116,298.15,-3.871601,0.910945,0.089055,0.000046,-4.333014,0.0,1.336000e-02,-1.874194,0.0,...,1.0,0.0,0.0,0.0,1.000000,3.0,2.754888,1.632993,0.040221,-0.396664
144117,313.15,-1.851552,0.567800,0.432200,0.000821,-3.085657,0.0,3.148800e-02,-1.501855,0.0,...,1.0,0.0,0.0,0.0,0.666667,4.0,26.264663,2.803039,0.126268,-0.300344
144118,293.15,-2.674074,0.308114,0.691886,0.000108,-3.967381,0.0,2.317000e-03,-2.635074,0.0,...,1.0,1.0,0.0,0.0,0.666667,5.0,33.874303,2.825749,0.208685,-0.351387


In [19]:
df_test = df2.iloc[idx_test].reset_index(drop=True)
df_test

,Temperature_K,LogS(mole_fraction),x1,x2,S1_pure_solubility,S1_pure_logS,S1_temp_diff,S2_pure_solubility,S2_pure_logS,S2_temp_diff,...,S2_NumHAcceptors,S2_NumRotatableBonds,S2_RingCount,S2_NumAromaticRings,S2_FractionCSP3,S2_HeavyAtomCount,S2_BertzCT,S2_BalabanJ,S2_MaxPartialCharge,S2_MinPartialCharge
0,283.15,-5.275560,0.022214,0.977786,0.1201,-0.920457,0.0,3.170000e-07,-6.498941,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
1,288.15,-5.162348,0.022214,0.977786,0.1392,-0.856361,0.0,4.024000e-07,-6.395342,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
2,293.15,-5.047498,0.022214,0.977786,0.1604,-0.794796,0.0,5.234000e-07,-6.281166,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
3,298.15,-4.938548,0.022214,0.977786,0.1845,-0.734004,0.0,6.683000e-07,-6.175029,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
4,303.15,-4.830914,0.022214,0.977786,0.2102,-0.677367,0.0,8.379000e-07,-6.076808,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20852,298.15,-1.675718,0.500000,0.500000,0.0275,-1.560667,0.0,1.861143e-04,-3.731865,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
20853,298.15,-1.595166,0.600000,0.400000,0.0275,-1.560667,0.0,1.861143e-04,-3.731865,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
20854,298.15,-1.518557,0.700000,0.300000,0.0275,-1.560667,0.0,1.861143e-04,-3.731865,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151
20855,298.15,-1.454693,0.800000,0.200000,0.0275,-1.560667,0.0,1.861143e-04,-3.731865,0.0,...,0,0,0,0,0.0,1,0.0,0.0,-0.41151,-0.41151


In [24]:
df_train_ex.drop(target_col, axis=1).to_csv('./MixtureSolDB_train_ex.csv', index=False)

In [26]:
np.savez("y", df_train_ex[target_col].values)

### Загружаем

In [27]:
df_train_ex = pd.read_csv('./MixtureSolDB_train_ex.csv')
y = np.load("y.npz")['arr_0']

In [28]:
y

array([-3.8884014751, -0.8968807465, -5.667157733 , -2.1817741064, -2.5622494372, -1.5386515664, -2.8181564121,
       -2.1558957693, -0.8532519864, -0.7848649545, -1.6665527255, -3.0159229661, -1.7632109006, -0.6925039621,
       -2.2781893848, -1.3590219426, -3.9863203027, -1.7675121336, -1.1278437273, -3.7358908437, ...,
       -1.7589518493, -1.3553644962, -2.0985961732, -2.2189630614, -1.9706162223, -1.097072904 , -0.9100948886,
       -2.7520267336, -4.2545348313, -2.193820026 , -1.4222784755, -2.3306831194, -2.2164537177, -0.2143570163,
       -2.505845406 , -2.2757241304, -3.8716007313, -1.8515515965, -2.6740740442, -1.2186755443], shape=(144120,))

# Собственно, обучение

In [ ]:
from pysr import PySRRegressor
model = PySRRegressor(
    #binary_operators=["+", "-", "*", "/", "^"],
    binary_operators=["+", "-", "*"],
    unary_operators=["exp", "square", "log"],
    
    niterations=300,          # можно начать с 300
    populations=30,
    population_size=50,
    ncycles_per_iteration=300,
    maxsize=20,
    maxdepth=5,
    procs=8,                  # использует все ядра
    random_state=42,
    deterministic=False,      # <- отключаем детерминизм для многопоточности
    batching=False,
    turbo=True,
    verbosity=1,               # Уровень детализации вывода
    select_k_features=12, # !!!! не больше входных значений!!!!
    elementwise_loss="L2DistLoss()",
)

In [1]:
!which python

/home/gleb/miniforge3/envs/py312/bin/python


In [ ]:
model.fit(
    df_train,
    y
)